In [4]:
%load_ext autoreload
%autoreload 2

import os,sys
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
import util as yu
from util import *
import util_Nsgm as yu2

yu.setpath('processData')

ens='a'
tfs=[10,12,14,16,18]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import util_Nsgm as yu
Tpack=24; d_jk=2
path='/capstor/store/cscs/userlab/lp139/lyan/code/projectData/01_Nsgm/data/NST_f_cA2.09.48_Nsgm_tf=10-18.h5'
data=yu.load(path,d=d_jk)
Ncfg,Njk=data['cfgs'][1:]
print(f'Ncfg={Ncfg},Njk={Njk}')

import util as yu

loading: /capstor/store/cscs/userlab/lp139/lyan/code/projectData/01_Nsgm/data/NST_f_cA2.09.48_Nsgm_tf=10-18.h5
276/276: diags/Z3pt/srcs/Z3pt.h5_NJNpi-f-Nsrc4*1-Nsigma                               
Ncfg=1228,Njk=614


In [10]:
import util_Nsgm as yu
flags={
    'cc2pt':True, # average quantities related by complex conjugation for 2pt
    'cc3pt':True, # same for 3pt (Removal of vacuum expectation value requires 'cc2pt'=='cc3pt')
    'll2pt':True, # average irrep rows 'l1' and conjugated 'l2' (Parity breaking effect of tmQCD has been taken care of)
    'll3pt':True, # same for 3pt (This flag has no effect if 'll2pt'=True and spin-projection is done)
    'r2pt': True, # making 2ptMat real for the rest frame # 'll2pt' has to be real for this flag
    'remove_pi0VEV':True, 
    'remove_jVEV':True,
}

def op_remove_pi0(op):
    t=op.split(';')
    if t[-1] in ['p','n','n,pi+']:
        return False
    if t[3] not in ['N0pi0,a','N1pi0,a','N0sgm0,a','N1sgm0,a']:
        return False
    if t[3] in ['N0sgm0,a','N1sgm0,a']:
        t[3]='a'
        t[-1]={'p,sgm':'p'}[t[-1]]
        return ';'.join(t)
    if t[2] == 'G1u':
        t[2]='G1g'
        assert(t[3]=='N0pi0,a'); t[3]='a'
        assert(t[-1]=='p,pi0'); t[-1]='p'
        return ';'.join(t)

def get2pt_diag(opa,opb,diag):
    opab=f'{opa}_{opb}'
    if opab not in data['2pt'].keys() or diag not in data['2pt'][opab].keys():
        return 0
    res=data['2pt'][opab][diag].copy()
    if not flags['remove_pi0VEV']:
        return res
    if diag == 'pi0f-pi0i':
        if opa==opb and opa=='t;0,0,0;pi0':
            res -= (data['VEV']['pi0f']**2)[:,None]
        return res
    
    if 'pi0f' in diag.split('-'):
        t_opa=op_remove_pi0(opa)
        if t_opa != False:
            t=diag.split('-'); t.remove('pi0f'); t_diag='-'.join(t)
            res -= data['2pt'][f'{t_opa}_{opb}'][t_diag] * data['VEV']['pi0f'][opa.split(';')[-1].split(',')[-1]][:,None]
    if 'pi0i' in diag.split('-'):
        t_opb=op_remove_pi0(opb)
        if t_opb != False:
            t=diag.split('-'); t.remove('pi0i'); t_diag='-'.join(t)
            res -= data['2pt'][f'{opa}_{t_opb}'][t_diag] * np.conj(data['VEV']['pi0f'][opb.split(';')[-1].split(',')[-1]][:,None])
    if 'pi0f' in diag.split('-') and 'pi0i' in diag.split('-'):
        t_opa=op_remove_pi0(opa); t_opb=op_remove_pi0(opb)
        if t_opa != False and t_opb != False:
            t=diag.split('-'); t.remove('pi0f'); t.remove('pi0i'); t_diag='-'.join(t)
            res += data['2pt'][f'{t_opa}_{t_opb}'][t_diag] * data['VEV']['pi0f'][opa.split(';')[-1].split(',')[-1]][:,None] * np.conj(data['VEV']['pi0f'][opb.split(';')[-1].split(',')[-1]][:,None])
    return res

def get2pt(opa,opb,diags=yu.diags_all):
    res=np.zeros([Njk,Tpack],dtype=complex)
    opab=f'{opa}_{opb}'; opba=f'{opb}_{opa}'
    if opab in data['2pt']:
        res+=np.sum([get2pt_diag(opa,opb,diag) for diag in data['2pt'][opab].keys() if diag in diags],axis=0)
    if opba in data['2pt']:
        diags_cc={'T', 'T-pi0f'}; 
        res+=np.conj(np.sum([get2pt_diag(opa,opb,diag) for diag in data['2pt'][opba].keys() if diag in diags_cc and diag in diags],axis=0))
    return res

def get2ptMat(ops,diags=yu.diags_all):
    if flags['ll2pt']:
        flags['ll2pt']=False
        ops_flip=[yu.op_flipl(op) for op in ops]
        t=(get2ptMat(ops,diags=diags)+np.conj(get2ptMat(ops_flip,diags=diags)))/2
        flags['ll2pt']=True
        if ops[0].split(';')[1]=='0,0,0' and flags['r2pt']:
            t=np.real(t)
        return t
    t=np.transpose([[get2pt(opa,opb,diags) for opb in ops] for opa in ops],[2,3,0,1])
    if flags['cc2pt']:
        t=(t+np.conj(np.transpose(t,[0,1,3,2])))/2
    return t

# tfs=[8,10,12,14,16,18,20]

def get3pt_diag(opa,opb,insert,diag):
    # if opa in ['g;0,0,0;G1g;a;l1;p','g;0,0,0;G1g;a;l2;p'] and opb in ['g;0,0,0;G1g;a;l1;p','g;0,0,0;G1g;a;l2;p'] and insert.startswith('id_j+_') and diag=='NJN':
    #     gm,j,tf=insert.split('_')
    #     res=c3ptDic_NJN[int(tf)]
    #     return res
    
    opab=f'{opa}_{opb}'
    if opab not in data['3pt'].keys() or diag not in data['3pt'][opab][insert].keys():
        return 0
    res=data['3pt'][opab][insert][diag].copy()
    if flags['remove_pi0VEV']:
        if 'pi0f' in diag.split('-'):
            t_opa=op_remove_pi0(opa)
            if t_opa != False:
                t=diag.split('-'); t.remove('pi0f'); t_diag='-'.join(t)
                res -= data['3pt'][f'{t_opa}_{opb}'][insert][t_diag] * data['VEV']['pi0f'][opa.split(';')[-1].split(',')[-1]][:,None]
        if 'pi0i' in diag.split('-'):
            t_opb=op_remove_pi0(opb)
            if t_opb != False:
                t=diag.split('-'); t.remove('pi0i'); t_diag='-'.join(t)
                res -= data['3pt'][f'{opa}_{t_opb}'][insert][t_diag] * np.conj(data['VEV']['pi0f'][opb.split(';')[-1].split(',')[-1]][:,None])
        if 'pi0f' in diag.split('-') and 'pi0i' in diag.split('-'):
            t_opa=op_remove_pi0(opa); t_opb=op_remove_pi0(opb)
            if t_opa != False and t_opb != False:
                t=diag.split('-'); t.remove('pi0f'); t.remove('pi0i'); t_diag='-'.join(t)
                res += data['3pt'][f'{t_opa}_{t_opb}'][insert][t_diag] * data['VEV']['pi0f'][opa.split(';')[-1].split(',')[-1]][:,None] * np.conj(data['VEV']['pi0f'][opb.split(';')[-1].split(',')[-1]][:,None])
    if flags['remove_jVEV']:
        gm,j,tf=insert.split('_')
        t_insert='_'.join([gm,j])
        if 'j' in diag.split('-') and t_insert in ['id_j+','id_js','id_jc','g5_j-']:
            t=diag.split('-'); t.remove('j'); t_diag='-'.join(t)
            res -= (get2pt_diag(opa,opb,t_diag)[:,int(tf)] * data['VEV']['j'][t_insert])[:,None]
    return res

def get3pt(opa,opb,insert,diags=yu.diags_all):
    res=np.zeros([Njk,int(insert.split('_')[-1])+1],dtype=complex)
    opab=f'{opa}_{opb}'; opba=f'{opb}_{opa}'
    if opab in data['3pt']:
        res+=np.sum([get3pt_diag(opa,opb,insert,diag) for diag in data['3pt'][opab][insert].keys() if diag in diags],axis=0)
    if opba in data['3pt']:
        t=data['3pt'][opb+'_'+opa][insert]; 
        diags_cc={'B3pt','W3pt','Z3pt','T-j', 'T-pi0f-j','T-jPf','B3pt-pi0f','W3pt-pi0f','Z3pt-pi0f'}; 
        if opab not in data['3pt'] or 'NJN-pi0f' not in data['3pt'][opab][insert]:
            diags_cc.add('NJN-pi0i')
        t_add=np.zeros([Njk,int(insert.split('_')[-1])+1],dtype=complex)+np.sum([get3pt_diag(opb,opa,insert,diag) for diag in t.keys() if diag in diags_cc and diag in diags],axis=0)
        res+=np.conj(t_add[:,::-1])*(yu.gtCj[insert.split('_')[0]])
    return res

def get3ptMat(opas,opbs,insert,diags=yu.diags_all):
    if flags['ll3pt']:
        flags['ll3pt']=False
        opas_flip=[yu.op_flipl(op) for op in opas]; opbs_flip=[yu.op_flipl(op) for op in opbs]
        sgns=np.array([[yu.op_getl_sgn(opa)*yu.op_getl_sgn(opb) for opb in opbs] for opa in opas])
        sgns*=yu.fourCPTstar[insert.split('_')[0]]
        t=(get3ptMat(opas,opbs,insert,diags=diags)+np.conj(get3ptMat(opas_flip,opbs_flip,insert,diags=diags))*sgns[None,None,:,:])/2
        flags['ll3pt']=True
        return t
    t=np.transpose([[get3pt(opa,opb,insert,diags) for opb in opbs] for opa in opas],[2,3,0,1])
    if flags['cc3pt']:
        flags['cc3pt']=False
        tt=get3ptMat(opbs,opas,insert,diags)[:,::-1]*(yu.gtCj[insert.split('_')[0]])
        flags['cc3pt']=True
        t=(t+np.conj(np.transpose(tt,[0,1,3,2])))/2
    return t


ops=['g;0,0,0;G1g;a;l1;p','g;0,0,0;G1g;N0sgm0,a;l1;p,sgm']
j='j+'

c2ptM=get2ptMat(ops)
tf2c3ptM={tf:get3ptMat(ops,ops,f'id_{j}_{tf}',diags=yu.diags_all) for tf in tfs}
tf2c3ptM_conn={tf:get3ptMat(ops,ops,f'id_{j}_{tf}',diags=yu.diags_jLoopless) for tf in tfs}
tf2c3ptM_disc={tf:get3ptMat(ops,ops,f'id_{j}_{tf}',diags=yu.diags_jLoopful) for tf in tfs}

import util as yu
c2ptCorrDic_NJN={tf:np.real(c2ptM[:,tf,0,0]) for tf in tfs}
yu.save_pkl_reg('data',[c2ptM,tf2c3ptM,c2ptCorrDic_NJN])

In [9]:
path='/capstor/store/cscs/userlab/lp139/lyan/code/projectData/NST_all_data/NST_f_cA2.09.48_meson2pt.h5'
with h5py.File(path) as f:
    # print(f['diags/P/data'].keys())
    # print(f['diags/pi0f-pi0i/data'])
    # print(f['VEV/pi0f/data'].keys())
    
    cfgs_m2=[cfg.decode() for cfg in f['cfgs'][:]]
    
    imom=0
    c2pt_piC=yu.jackknife(f['diags/P/data/pi+_pi+'][:,:,imom],d=d_jk)
    c2pt_pi0=yu.jackknife(f['diags/P/data/pi0_pi0'][:,:,imom]+f['diags/pi0f-pi0i/data/pi0_pi0'][:,:,imom],d=d_jk)
    c2pt_pi0_conn=yu.jackknife(f['diags/P/data/pi0_pi0'][:,:,imom],d=d_jk)
    c2pt_sgm=yu.jackknife(f['diags/P/data/sgm_sgm'][:,:,imom]+f['diags/pi0f-pi0i/data/sgm_sgm'][:,:,imom],d=d_jk)
    c2pt_sgm_conn=yu.jackknife(f['diags/P/data/sgm_sgm'][:,:,imom],d=d_jk)
    
    vev_pi0=yu.jackknife(f['VEV/pi0f/data/pi0'][:],d=d_jk)
    vev_sgm=yu.jackknife(f['VEV/pi0f/data/sgm'][:],d=d_jk)
    
    if True: # symmetrize
        func=lambda t:(t+np.roll(np.flip(t,axis=1),1,axis=1))/2
        c2pt_piC=func(c2pt_piC)
        c2pt_pi0=func(c2pt_pi0)
        c2pt_sgm=func(c2pt_sgm)
        c2pt_pi0_conn=func(c2pt_pi0_conn)
        
    if True: # vev subtract
        c2pt_pi0 -= vev_pi0[:,None]**2
        c2pt_sgm -= vev_sgm[:,None]**2
        
    if True: # making real
        c2pt_piC=np.real(c2pt_piC)
        c2pt_pi0=np.real(c2pt_pi0)
        c2pt_sgm=np.real(c2pt_sgm)
        c2pt_pi0_conn=np.real(c2pt_pi0_conn)

Ncfg=len(cfgs_m2); Njk=len(c2pt_piC)
print(f'Ncfg={Ncfg}, Njk={Njk}')

data_meson2pt=[c2pt_piC,c2pt_pi0,c2pt_pi0_conn,c2pt_sgm,c2pt_sgm_conn]
yu.save_pkl_reg('data_meson2pt',data_meson2pt)

Ncfg=1228, Njk=614
